# Lesson 11: Practical Implementation & Open-Source MCP API Gateways

In this notebook, you will:
1. Build an open-source **FastAPI Reverse-Proxy MCP Gateway**.
2. Enforce `Mcp-Method` weighted token-bucket rate limiting and audit logging.
3. Test gateway responses using `TestClient`.

In [ ]:
!pip install -q -U fastapi httpx

In [ ]:
import time
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

app = FastAPI()
BUCKETS = {}
CAPACITY, REFILL = 4.0, 1.0

@app.post("/mcp")
async def mcp_gateway(request: Request):
    tenant = request.headers.get("X-Tenant-ID", "anon")
    method = request.headers.get("Mcp-Method", "tools/call")
    cost = 1.0 if method == "tools/list" else 2.0
    now = time.monotonic()
    tokens, last = BUCKETS.get(tenant, (CAPACITY, now))
    tokens = min(CAPACITY, tokens + REFILL * (now - last))
    if tokens < cost:
        return JSONResponse(status_code=429, content={"error": "Rate limit exceeded"})
    BUCKETS[tenant] = (tokens - cost, now)
    body = await request.json()
    return {"jsonrpc": "2.0", "id": body.get("id"), "result": {"gateway_ok": True, "remaining": round(tokens - cost, 2)}}

client = TestClient(app)
for i in range(4):
    r = client.post("/mcp", headers={"X-Tenant-ID": "t1", "Mcp-Method": "tools/call"}, json={"jsonrpc": "2.0", "id": i})
    print(f"Call {i+1}: status={r.status_code}, body={r.json()}")

### Challenge Exercise
Add a header-to-body integrity check to `mcp_gateway()` that returns HTTP `400` if `request.headers['Mcp-Method'] != body.get('method')`.